In [ ]:
import os
import pandas as pd
import commons as c

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.metrics import precision_score, recall_score, f1_score

# Boxplots of mutant detectability (distance between mutant and original)


In [ ]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)
df_normal['nature'] = 'non-equivalent'

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)
df_equiv['nature'] = 'equivalent'


In [ ]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df_total = df.melt(
    id_vars=['Algorithm', 'Qubits_number', 'hardware', 'nature', 'metric', 'metric_full', 'hardware_named'],
    value_vars=['ideal_distance', 'noisy_distance'],
    value_name='distance'
)
df_total.loc[df_total['variable'] == 'ideal_distance', 'hardware'] = 'ideal'
df_total.loc[df_total['variable'] == 'ideal_distance', 'hardware_named'] = 'Noiseless'
df_total = df_total.drop(columns=['variable'])

# RQ1.1: 

In [ ]:
def print_box_plot(df, file_name):

    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(df, 
            y='distance', 
            x="metric_full", 
            color='hardware_named',  
            # title="Boxplot of Distance by noise model and program type",
            labels={"metric_full": "Metric", 'hardware_named': "Simulator", "distance": "Distance"},
            points=False,
            boxmode="group",
            color_discrete_map=c.color_map
    )

    # Save the figure
    output_folder = 'results/RQ1/RQ1_1/'
    os.makedirs(output_folder, exist_ok=True)
    c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1])

In [ ]:
fig_name ="Visualisation of the mutant detectability score for each metric"

df_equiv = df_total[df_total['nature'] == 'equivalent']
file_name = f'visu_equiv'
print_box_plot(df_equiv, file_name)
  
df_normal = df_total[df_total['nature'] == 'non-equivalent']  
file_name = f'visu_normal'
print_box_plot(df_normal, file_name)    

# RQ1.2: 

In [ ]:
def add_threshold_line(fig, hw, t, metric, xmin, xmax, color, dash):
    
    threshold_value = c.get_tolerance_values(hw, t)[metric]
    fig.add_shape(
        type="line",
        x0= xmin-0.5,
        x1= xmax+0.5,
        y0=threshold_value,
        y1=threshold_value,
        line=dict(color=color, dash=dash, width=4),
        xref="x",
        yref="y",
    )
        
    if t == 'N':
        fig.add_trace(go.Scatter(
            x=[None], y=[None],  # Invisible points
            mode="lines",
            line=dict(color=color, dash=dash),
            name=f"Threshold N<sub>{hw}</sub>" #N_{hw}"
        ))
    else:
        fig.add_trace(go.Scatter(
            x=[None], y=[None],  # Invisible points
            mode="lines",
            line=dict(color=color, dash=dash),
            name=f"Threshold {t}"
        ))


In [ ]:
def print_box_plot(df, file_name, metric, with_threshold=False):

    df = df.copy()  # Ensure it's a copy
    
    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['nature'] = df['nature'].map(label_mapping)
    
    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(df, 
            y='distance', 
            x="hardware_named", 
            color='nature',  
            # title="Boxplot of distance by noise model and mutation type",
            labels={"hardware_named": "Simulator", 'noisy_distance': "Distance", 'nature': "Mutant type"},
            points=False,
            boxmode="group")
    
    # Add the threshold line
    if with_threshold:
        colors = px.colors.qualitative.Prism
        add_threshold_line(fig, '', 'I', metric, 0, 3, colors[4], None)
        add_threshold_line(fig, '', 'M', metric, 1, 3, colors[3], None)
        
        for idx, hw in enumerate(c.hardware):
            add_threshold_line(fig, hw, 'N', metric, idx+1, idx+1, colors[2], None)
    
        add_threshold_line(fig, '', 'A', metric, 1, 3, colors[1], None)
        output_folder = 'results/RQ1/RQ1_2/with_thresholds/'
    
    else:   
        output_folder = 'results/RQ1/RQ1_2/no_thresholds/'
        
    # Save the figure
    os.makedirs(output_folder, exist_ok=True)
    c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1], height=500)


In [ ]:
for m in c.metrics:
    df_metric = df_total[df_total['metric'] == m]
    fig_name = f"Comparison between equivalent and non-equivalent mutant detectability for {m}"
    file_name = f'visu_{m}'
    print_box_plot(df_metric, file_name, c.metrics[m])
    print_box_plot(df_metric, file_name, c.metrics[m], True)

# RQ1.3: Confusion matrices


In [ ]:
def confusion_matrix(df, col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [ ]:
# Define a function to create a heatmap with annotations
def create_heatmap(fig, data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale= [[0.0, '#eff3ff'], [0.05, '#9ecae1'],[0.1, '#6baed6'], [0.8, '#3182bd'], [1, '#08519c']],
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}',  # Format the text annotations
            textfont=dict(size=40)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    

In [ ]:
def print_confusion_matrices(df_confusion, threshold, mutant, hw):
    # Create subplots with titles
    
    fig = make_subplots(
        rows=1, cols=5,
        subplot_titles=list(c.metric_names.values()),
        x_title='Noisy', y_title='Ideal', horizontal_spacing=0.05
    )
    
    fig.update_layout(annotations=[dict(font=dict(size=25))])

    print(f'Confusion matrix for {threshold} and {hw}')
    
    for j, metric in enumerate(c.metrics):
        matrix = confusion_matrix(df_confusion, 'Killed_I' + metric, 'Killed_N' + metric)
        # Add heatmaps to subplots
        create_heatmap(fig, matrix, row=1, col=j+1, showscale=True)
    
    folder_name = "results/RQ1/RQ1_3/"
    c.setup_layout_and_save(fig, folder_name, f"{mutant}_{threshold}_{hw}", height=550, width=3000)


In [ ]:
type_dict = ({
    "gates": int,
    "depth": int,
    "singlequbit_gates": int,
    "multiqubit_gates": int,
    "Input": str,
    "Ideal_hellinger": float,
    "Noisy_hellinger": float,
    "Ideal_jensenshannon": float,
    "Noisy_jensenshannon": float,
    "Ideal_trace": float,
    "Noisy_trace": float,
    "Ideal_fidelity": float,
    "Noisy_fidelity": float,
    "Ideal_expectation": float,
    "Noisy_expectation": float,
    "Killed_IH": bool,
    "Killed_NH": bool,
    "Killed_IJ": bool,
    "Killed_NJ": bool,
    "Killed_IT": bool,
    "Killed_NT": bool,
    "Killed_IF": bool,
    "Killed_NF": bool,
    "Killed_IE": bool,
    "Killed_NE": bool,
    "Input_type": str,
    "Algorithm": str,
    "Qubits_number": int,
    "Operator": str,
    "Gate": str,
    "Position": int,
    "Qubits": int,
    "Params": float,
    "Gate_type": str,
    "Relative_position": str,
    "Output_type": str
})

for threshold in c.thresholds:
    for mutant_type in ["balanced"]: #c.mutant_types:
        for hw in c.hardware:
            csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
            df_confusion = pd.read_csv(csv_path, dtype=type_dict)
            print_confusion_matrices(df_confusion, threshold, mutant_type, hw)

 # RQ1.4: F1, Precision and recall

In [ ]:
def get_scores(df):
    results = []
    
    for metric in c.metrics.keys():
        
        true_labels = df[f'Killed_I{metric}']
        predicted_labels = df[f'Killed_N{metric}']
        
        # Calculate precision, recall, and F1 score for the pair
        precision = precision_score(true_labels, predicted_labels, zero_division=0)
        recall = recall_score(true_labels, predicted_labels, zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, zero_division=0)
        results.append((precision, recall, f1))
    
    scores_df = pd.DataFrame(results, columns=['Precision', 'Recall', 'F1 Score'], 
                             index=[f'{metric}' for metric in c.metrics])
    
    return scores_df.round(4)


In [ ]:
output_folder = 'results/RQ1/RQ1_4'
os.makedirs(output_folder, exist_ok=True)

scores = ["Precision", "Recall", "F1 Score"]

mutant_type = "balanced"

# Initialize dictionaries to store score DataFrames with (metric, threshold) index
dic_scores = {
    score: pd.DataFrame(columns=c.hardware, index=pd.MultiIndex.from_product(
        [[metric for metric in c.metrics.keys()], c.thresholds], names=["Metric", "Threshold"]
    ))
    for score in scores
}
    
for threshold in c.thresholds:
    for hw in c.hardware:
        csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
        df = pd.read_csv(csv_path)
        scores_df = get_scores(df)
            
        for score in scores:
            for metric in scores_df.index:
                dic_scores[score].at[(metric, threshold), hw] = scores_df.at[metric, score]

# Save each score DataFrame as a CSV
for score, df_score in dic_scores.items():
    
    output_path = os.path.join(output_folder, f'{mutant_type}_{score}.csv')
    df_score.to_csv(output_path)
    print(f"Saved {output_path}")
